In [2]:
from pathlib import Path

import duckdb

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
EC = ROOT / "data/kaggle/e-commerce_dataset"
MF = ROOT / "data/kaggle/marketing_funnel"
print(ROOT)

duckdb.sql(
    f"SUMMARIZE SELECT * FROM '{MF / 'olist_marketing_qualified_leads_dataset.csv'}'"
).show()

/Users/dmzzz/Desktop/Marketing_Services/02_Training/marketing-roi-pipeline
┌────────────────────┬─────────────┬──────────────────────────────────┬──────────────────────────────────┬───────────────┬───────────────────────┬───────┬────────────┬────────────┬────────────┬───────┬─────────────────┐
│    column_name     │ column_type │               min                │               max                │ approx_unique │          avg          │  std  │    q25     │    q50     │    q75     │ count │ null_percentage │
│      varchar       │   varchar   │             varchar              │             varchar              │     int64     │        varchar        │ int32 │  varchar   │  varchar   │  varchar   │ int64 │  decimal(9,2)   │
├────────────────────┼─────────────┼──────────────────────────────────┼──────────────────────────────────┼───────────────┼───────────────────────┼───────┼────────────┼────────────┼────────────┼───────┼─────────────────┤
│ mql_id             │ VARCHAR     │ 0002ac0d

In [3]:
LEADS = MF / "olist_marketing_qualified_leads_dataset.csv"
duckdb.sql(
    f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT mql_id) AS n_ids FROM '{LEADS}'"
).show()
duckdb.sql(
    f"SELECT origin, COUNT(*) AS n FROM '{LEADS}' GROUP BY 1 ORDER BY n DESC"
).show()
duckdb.sql(
    f"SELECT date_trunc('month', first_contact_date) AS month, COUNT(*) AS n FROM '{LEADS}' GROUP BY 1 ORDER BY 1"
).show()

┌────────┬───────┐
│ n_rows │ n_ids │
│ int64  │ int64 │
├────────┼───────┤
│   8000 │  8000 │
└────────┴───────┘

┌───────────────────┬───────┐
│      origin       │   n   │
│      varchar      │ int64 │
├───────────────────┼───────┤
│ organic_search    │  2296 │
│ paid_search       │  1586 │
│ social            │  1350 │
│ unknown           │  1099 │
│ direct_traffic    │   499 │
│ email             │   493 │
│ referral          │   284 │
│ other             │   150 │
│ display           │   118 │
│ other_publicities │    65 │
│ NULL              │    60 │
└───────────────────┴───────┘
  11 rows         2 columns

┌─────────────────────┬───────┐
│        month        │   n   │
│      timestamp      │ int64 │
├─────────────────────┼───────┤
│ 2017-06-01 00:00:00 │     4 │
│ 2017-07-01 00:00:00 │   239 │
│ 2017-08-01 00:00:00 │   386 │
│ 2017-09-01 00:00:00 │   312 │
│ 2017-10-01 00:00:00 │   416 │
│ 2017-11-01 00:00:00 │   445 │
│ 2017-12-01 00:00:00 │   200 │
│ 2018-01-01 00:00:00 │ 

In [7]:
duckdb.sql(f"SUMMARIZE SELECT * FROM '{MF / 'olist_closed_deals_dataset.csv'}'").show()

┌───────────────────────────────┬─────────────┬──────────────────────────────────┬──────────────────────────────────┬───────────────┬────────────────────────────┬────────────────────┬────────────────────────────┬────────────────────────────┬─────────────────────────┬───────┬─────────────────┐
│          column_name          │ column_type │               min                │               max                │ approx_unique │            avg             │        std         │            q25             │            q50             │           q75           │ count │ null_percentage │
│            varchar            │   varchar   │             varchar              │             varchar              │     int64     │          varchar           │      varchar       │          varchar           │          varchar           │         varchar         │ int64 │  decimal(9,2)   │
├───────────────────────────────┼─────────────┼──────────────────────────────────┼──────────────────────────────────┼─

In [8]:
DEALS = MF / "olist_closed_deals_dataset.csv"
duckdb.sql(
    f"SELECT AVG(CASE WHEN declared_monthly_revenue = 0 THEN 1 ELSE 0 END) AS share_zero FROM '{DEALS}'"
).show()

┌────────────────────┐
│     share_zero     │
│       double       │
├────────────────────┤
│ 0.9465558194774347 │
└────────────────────┘



In [9]:
duckdb.sql(
    f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT seller_id) AS n_sellers FROM '{DEALS}'"
).show()

┌────────┬───────────┐
│ n_rows │ n_sellers │
│ int64  │   int64   │
├────────┼───────────┤
│    842 │       842 │
└────────┴───────────┘



In [10]:
ITEMS = EC / "olist_order_items_dataset.csv"
duckdb.sql(f"""
    SELECT
        COUNT(*)                                        AS deals,
        COUNT(s.seller_id)                              AS matched,
        ROUND(100.0 * COUNT(s.seller_id) / COUNT(*), 1) AS match_pct
    FROM '{DEALS}' d
    LEFT JOIN (SELECT DISTINCT seller_id FROM '{ITEMS}') s USING (seller_id)
""").show()

┌───────┬─────────┬───────────┐
│ deals │ matched │ match_pct │
│ int64 │  int64  │  double   │
├───────┼─────────┼───────────┤
│   842 │     380 │      45.1 │
└───────┴─────────┴───────────┘

